# Student Performance Prediction — Complete Google Colab Notebook

**Beginner-friendly | Python | Supervised Machine Learning | Approx. 30–45 minutes including review**

## What this project does
This notebook builds two models:
1. **Regression:** predicts a student's final exam score (0–100).
2. **Classification:** predicts a performance category (`Needs Improvement`, `Average`, `Good`, `Excellent`).

It includes dataset creation, data inspection, cleaning, exploratory data analysis (EDA), model training, evaluation, graphs, and a prediction for a new student.

### Time plan
| Activity | Approx. time |
|---|---:|
| Setup and dataset | 5 min |
| Inspect and clean data | 5 min |
| EDA and graphs | 7 min |
| Train and compare models | 10 min |
| Evaluate and test a new student | 8 min |
| Review and understand results | 5–10 min |

**Execution note:** The code normally runs much faster than 30–45 minutes on standard Colab CPU. The 30–45 minute estimate includes reading explanations and reviewing outputs. No GPU is required.

### Important dataset note
To make the notebook run immediately without manual file uploads, this project generates a **synthetic educational dataset** of 1,000 sample student records. It is fictional demonstration data, not real student information. Do not describe it as collected university data or use its scores to make decisions about real students.


## 1. Import required libraries
We use pandas and NumPy for data handling, Matplotlib and Seaborn for graphs, and scikit-learn for machine learning.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print("All libraries imported successfully.")


## 2. Create the student dataset
We generate 1,000 reproducible sample records. Each row represents one fictional student. The final score is calculated using a demonstration formula plus random variation.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 1000

study_hours = np.clip(rng.normal(3.5, 1.5, n), 0.2, 10)
attendance = np.clip(rng.normal(82, 11, n), 40, 100)
previous_score = np.clip(rng.normal(68, 14, n), 10, 100)
assignments = rng.integers(0, 11, n)
sleep_hours = np.clip(rng.normal(7, 1.1, n), 3, 10)
extracurricular = rng.choice(["Yes", "No"], n, p=[0.42, 0.58])
internet_access = rng.choice(["Yes", "No"], n, p=[0.88, 0.12])
parental_support = rng.choice(["High", "Medium", "Low"], n, p=[0.35, 0.45, 0.20])
distance_km = np.clip(rng.gamma(2.0, 3.0, n), 0.2, 25)

extra_num = (extracurricular == "Yes").astype(int)
internet_num = (internet_access == "Yes").astype(int)
support_num = pd.Series(parental_support).map({"Low": 0, "Medium": 1, "High": 2}).to_numpy()

noise = rng.normal(0, 7, n)
final_score = (
    20 + 3.4*study_hours + 0.20*attendance + 0.22*previous_score
    + 1.25*assignments + 1.2*sleep_hours + 2.0*extra_num
    + 2.0*internet_num + 2.2*support_num - 0.22*distance_km + noise
)
final_score = np.clip(final_score, 0, 100).round(1)

performance_level = pd.cut(
    final_score,
    bins=[-0.1, 39.9, 59.9, 74.9, 100],
    labels=["Needs Improvement", "Average", "Good", "Excellent"]
).astype(str)

df = pd.DataFrame({
    "study_hours_per_day": study_hours.round(1),
    "attendance_percentage": attendance.round(1),
    "previous_exam_score": previous_score.round(1),
    "assignments_completed": assignments,
    "sleep_hours_per_day": sleep_hours.round(1),
    "extracurricular_activities": extracurricular,
    "internet_access": internet_access,
    "parental_support": parental_support,
    "distance_from_school_km": distance_km.round(1),
    "final_score": final_score,
    "performance_level": performance_level
})

# Save a copy of the generated dataset to the Colab session.
df.to_csv("student_performance_dataset.csv", index=False)

print("Dataset created successfully!")
print("Number of student records:", len(df))
print("Number of columns:", len(df.columns))
display(df.head(10))


## 3. Understand the dataset
`head()` displays the first records, `shape` gives rows and columns, and `info()` shows data types and non-null values.

In [ ]:
print("Dataset dimensions:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe().round(2))


## 4. Check missing values and duplicates
We check the dataset quality. This generated dataset should contain no missing values or duplicate rows, but the checks are included as good practice.

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing values"))

print("Number of duplicate rows:", df.duplicated().sum())

# Cleaning step: remove duplicates and rows without target values.
df = df.drop_duplicates().dropna(subset=["final_score", "performance_level"]).copy()
print("Shape after cleaning:", df.shape)


## 5. Exploratory Data Analysis (EDA)
EDA helps us understand patterns before training a model. Run each cell to generate the graphs.

### Graph 1 — Distribution of final scores
A histogram shows how student scores are distributed.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["final_score"], bins=20, kde=True)
plt.title("Distribution of Student Final Scores")
plt.xlabel("Final score")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

print("Average final score:", round(df["final_score"].mean(), 2))
print("Minimum final score:", df["final_score"].min())
print("Maximum final score:", df["final_score"].max())


### Graph 2 — Count of students in each performance category
A count plot compares the number of records in each category.

In [ ]:
category_order = ["Needs Improvement", "Average", "Good", "Excellent"]
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="performance_level", order=category_order)
plt.title("Student Count by Performance Category")
plt.xlabel("Performance category")
plt.ylabel("Number of students")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(df["performance_level"].value_counts().reindex(category_order).to_frame("Student count"))


### Graph 3 — Study hours vs final score
This scatter plot shows the relationship between daily study hours and final scores. A relationship in this sample does not prove causation.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="study_hours_per_day", y="final_score", alpha=0.6)
plt.title("Daily Study Hours vs Final Exam Score")
plt.xlabel("Study hours per day")
plt.ylabel("Final score")
plt.tight_layout()
plt.show()


### Graph 4 — Correlation heatmap
Correlation measures the strength and direction of linear relationships between numeric variables. It does not prove cause and effect.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", square=False)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()


## 6. Prepare input features and target variables

We build two separate prediction tasks:

- **Regression target:** `final_score` — a numeric value.
- **Classification target:** `performance_level` — a category.

We remove both target columns from the input features so that the model cannot simply read the answer. We use a stratified split for classification to keep category proportions similar in train and test sets.


In [ ]:
TARGET_REG = "final_score"
TARGET_CLASS = "performance_level"

X = df.drop(columns=[TARGET_REG, TARGET_CLASS])
y_reg = df[TARGET_REG]
y_class = df[TARGET_CLASS]

# Regression split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.20, random_state=RANDOM_STATE
)

# Classification split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_class, test_size=0.20, random_state=RANDOM_STATE, stratify=y_class
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Training records:", len(X_train_reg))
print("Testing records:", len(X_test_reg))
print("Numeric input columns:", numeric_features)
print("Categorical input columns:", categorical_features)


## 7. Preprocess the data

Machine learning algorithms need numeric input:
- Missing numeric values are filled with the median.
- Missing text/category values are filled with the most frequent category.
- Categorical columns are converted using one-hot encoding.
- Standard scaling is used for Linear Regression and Logistic Regression.

Preprocessing is inside a pipeline. This helps prevent **data leakage**, because the transformations are fitted only on training data.


In [ ]:
def create_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(steps=numeric_steps)
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

print("Preprocessing pipeline is ready.")


## 8. Train regression models — predict final exam score
We train three models and compare their test-set results. Lower MAE/RMSE and higher R² are generally preferred.

In [ ]:
regression_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
}

regression_results = []
trained_regression_models = {}

for model_name, model in regression_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", create_preprocessor(scale_numeric=(model_name == "Linear Regression"))),
        ("model", model)
    ])
    pipeline.fit(X_train_reg, y_train_reg)
    predictions = pipeline.predict(X_test_reg)

    regression_results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test_reg, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test_reg, predictions)),
        "R2": r2_score(y_test_reg, predictions)
    })
    trained_regression_models[model_name] = pipeline

regression_results_df = pd.DataFrame(regression_results).sort_values("RMSE").reset_index(drop=True)
display(regression_results_df.round(3))


### Graph 5 — Compare regression models by RMSE
A shorter bar indicates a smaller average squared-error-based prediction penalty.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=regression_results_df, x="Model", y="RMSE")
plt.title("Regression Model Comparison (RMSE)")
plt.ylabel("RMSE — lower is better")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_regression_name = regression_results_df.loc[0, "Model"]
best_regression_model = trained_regression_models[best_regression_name]
best_regression_predictions = best_regression_model.predict(X_test_reg)

print("Best regression model by RMSE:", best_regression_name)


### Graph 6 — Actual scores vs predicted scores
Points closer to the dashed diagonal represent predictions closer to the actual scores.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test_reg, best_regression_predictions, alpha=0.65)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.xlabel("Actual final score")
plt.ylabel("Predicted final score")
plt.title(f"Actual vs Predicted Scores — {best_regression_name}")
plt.tight_layout()
plt.show()

print("Regression evaluation:")
print("MAE :", round(mean_absolute_error(y_test_reg, best_regression_predictions), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_reg, best_regression_predictions)), 2))
print("R²  :", round(r2_score(y_test_reg, best_regression_predictions), 3))


## 9. Train classification models — predict performance category
We train Logistic Regression, Decision Tree, and Random Forest classifiers. Weighted precision, recall, and F1-score account for class sizes.

In [ ]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
}

classification_results = []
trained_classification_models = {}

for model_name, model in classification_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", create_preprocessor(scale_numeric=(model_name == "Logistic Regression"))),
        ("model", model)
    ])
    pipeline.fit(X_train_clf, y_train_clf)
    predictions = pipeline.predict(X_test_clf)

    classification_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_clf, predictions),
        "Precision_weighted": precision_score(y_test_clf, predictions, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_test_clf, predictions, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_test_clf, predictions, average="weighted", zero_division=0)
    })
    trained_classification_models[model_name] = pipeline

classification_results_df = pd.DataFrame(classification_results).sort_values(
    "F1_weighted", ascending=False
).reset_index(drop=True)

display(classification_results_df.round(3))


### Graph 7 — Compare classification models by weighted F1-score
F1 combines precision and recall. Higher is generally better.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=classification_results_df, x="Model", y="F1_weighted")
plt.title("Classification Model Comparison (Weighted F1)")
plt.ylabel("Weighted F1-score — higher is better")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_classification_name = classification_results_df.loc[0, "Model"]
best_classification_model = trained_classification_models[best_classification_name]
best_classification_predictions = best_classification_model.predict(X_test_clf)

print("Best classification model by weighted F1:", best_classification_name)


## 10. Classification report and confusion matrix
The classification report shows precision, recall, and F1-score for each class. The confusion matrix compares actual labels (rows) with predicted labels (columns).

In [ ]:
print(classification_report(
    y_test_clf,
    best_classification_predictions,
    labels=category_order,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test_clf,
    best_classification_predictions,
    labels=category_order,
    xticks_rotation=25,
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_classification_name}")
plt.tight_layout()
plt.show()


## 11. Predict the result for a new student

Change the example values below to try a different fictional student. The input column names and categories must match the training data.


In [ ]:
new_student = pd.DataFrame([{
    "study_hours_per_day": 4.0,
    "attendance_percentage": 85.0,
    "previous_exam_score": 70.0,
    "assignments_completed": 8,
    "sleep_hours_per_day": 7.0,
    "extracurricular_activities": "Yes",
    "internet_access": "Yes",
    "parental_support": "Medium",
    "distance_from_school_km": 4.0
}])

predicted_score = best_regression_model.predict(new_student)[0]
predicted_category = best_classification_model.predict(new_student)[0]

print("NEW STUDENT INPUT")
display(new_student)
print(f"Predicted final exam score: {predicted_score:.1f} / 100")
print(f"Predicted performance category: {predicted_category}")


## 12. Save the models, dataset, and result tables
These files will be saved in the current Colab session. You can download them from the Files panel on the left.

In [ ]:
joblib.dump(best_regression_model, "best_student_score_model.joblib")
joblib.dump(best_classification_model, "best_student_category_model.joblib")

df.to_csv("student_performance_dataset.csv", index=False)
regression_results_df.to_csv("regression_model_comparison.csv", index=False)
classification_results_df.to_csv("classification_model_comparison.csv", index=False)

print("Files saved:")
print("- student_performance_dataset.csv")
print("- best_student_score_model.joblib")
print("- best_student_category_model.joblib")
print("- regression_model_comparison.csv")
print("- classification_model_comparison.csv")

# Optional: uncomment to download files automatically.
# from google.colab import files
# files.download("student_performance_dataset.csv")
# files.download("best_student_score_model.joblib")
# files.download("best_student_category_model.joblib")
# files.download("regression_model_comparison.csv")
# files.download("classification_model_comparison.csv")


## 13. Final project summary and conclusion

### What we completed
- Generated and inspected a dataset containing 1,000 fictional student records.
- Checked missing values and duplicate rows.
- Created four EDA graphs.
- Preprocessed numeric and categorical features.
- Trained and compared three regression models.
- Trained and compared three classification models.
- Evaluated models with appropriate metrics and graphs.
- Predicted a new student's score and performance category.
- Saved the dataset, comparison tables, and trained models.

### How to explain the project
**Student Performance Prediction** is a supervised machine-learning project. It uses student-related features such as study hours, attendance, previous marks, assignment completion, and sleep hours to estimate a final exam score and classify a performance level.

Regression is used because a final score is numeric. Classification is used because performance level belongs to a fixed category. We compare Linear Regression, Decision Tree, Random Forest, and Logistic Regression models using a held-out test set.

### Limitations and responsible use
- This notebook uses synthetic demonstration data. Its model scores are not evidence of real-world accuracy.
- Real student performance depends on many factors not included here.
- Do not use these predictions alone to grade, penalize, admit, or make other high-impact decisions about students.
- A real deployment requires approved representative data, privacy protection, fairness testing, external validation, and educator oversight.

**Project completed!** Restart and run all cells to reproduce the results.
